#Simple ReAct Agent from Scratch

In [ ]:
# based on https://til.simonwillison.net/llms/python-react-pattern

In [ ]:
import os

# Optional: sanity check
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not loaded"

In [ ]:
import openai
import re
import httpx
import os
from dotenv import load_dotenv

_ = load_dotenv()
from openai import OpenAI

In [ ]:
client = OpenAI()

In [ ]:
chat_completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello world"}]
)

In [ ]:
chat_completion.choices[0].message.content

In [ ]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gpt-4o-mini", 
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content
    

In [ ]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

In [ ]:
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [ ]:
abot = Agent(prompt)

In [ ]:
result = abot("How much does a toy poodle weigh?")
print(result)

In [ ]:
result = average_dog_weight("Toy Poodle")

In [ ]:
result

In [ ]:
next_prompt = "Observation: {}".format(result)

In [ ]:
abot(next_prompt)

In [ ]:
abot.messages

In [ ]:
abot = Agent(prompt)

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
abot(question)

In [ ]:
next_prompt = "Observation: {}".format(average_dog_weight("Border Collie"))
print(next_prompt)

In [ ]:
abot(next_prompt)

In [ ]:
next_prompt = "Observation: {}".format(average_dog_weight("Scottish Terrier"))
print(next_prompt)

In [ ]:
abot(next_prompt)

In [ ]:
next_prompt = "Observation: {}".format(eval("37 + 20"))
print(next_prompt)

In [ ]:
abot(next_prompt)

### Add loop 

In [ ]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

In [ ]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a) 
            for a in result.split('\n') 
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

## SD Modifications

### Modify the provided IPYNB script to add one more Dog Breed that is not listed in the script

In [ ]:
# New function
def calculate(what):
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier": 
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    elif name in "Great Dane":
        return("a great dane average weight is 120 lbs")
    else:
        return("An average dog weights 50 lbs")

# Redoing the known actions to catch updated function
known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

In [ ]:
# Checking if the new breed is registered
print(average_dog_weight("Great Dane"))

### Modify the script to calculate the combined weight for 3 dogs (you choose any breeds), including the Dog Breed you added earlier

In [ ]:
sd_question = """I have 3 dogs, a great dane, a border collie and a scottish terrier. \
What is their combined weight"""
query(sd_question)

### SD Response to Module 2 Discussion

In [ ]:
disc2prompt = """Businesses are embracing AI agents to transform workflows, enhance efficiency, and reduce manual effort. These agents leverage LLMs for dynamic decision-making, workflow automation, and seamless collaboration with human teams and existing systems. Examine the role of AI agents in automating diverse business processes for 'Cloud Computing and eCommerce'. 

List and summarize the business opportunities in "Cloud Computing and eCommerce“ that can benefit from LLM powered AI Agents."""

In [ ]:
# | output: false
# | echo: false

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "Translate the following from English into {language}"

prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", "{text}")]
)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
response = model.invoke([HumanMessage(disc2prompt)])
print(response.content)

### Markdown response
AI agents powered by large language models (LLMs) are gaining traction across various sectors, including cloud computing and eCommerce. Here are some key roles they play and business opportunities they create:

### 1. **Customer Support Automation**
   - **Opportunity**: Implement AI-driven chatbots and virtual agents to handle customer inquiries around the clock.
   - **Benefit**: Reduces the need for large customer service teams, improves response times, and enhances customer satisfaction.

### 2. **Personalized Shopping Experiences**
   - **Opportunity**: Utilize AI agents to analyze user behavior and preferences for personalized product recommendations.
   - **Benefit**: Increases sales conversion rates and enhances customer retention by providing tailored experiences.

### 3. **Inventory and Supply Chain Management**
   - **Opportunity**: Automate inventory tracking and order processing using AI agents to predict stock levels and supply chain disruptions.
   - **Benefit**: Reduces costs, minimizes stockouts and overstock situations, and optimizes the supply chain.

### 4. **Dynamic Pricing Strategies**
   - **Opportunity**: Integrate AI agents that can analyze market trends, competitor pricing, and customer demand to adjust prices in real-time.
   - **Benefit**: Maximizes revenue opportunities and ensures competitiveness in the market.

### 5. **Content Generation for Marketing**
   - **Opportunity**: Deploy LLMs to create product descriptions, blogs, and marketing materials dynamically based on trends and consumer interests.
   - **Benefit**: Saves time and resources in content creation while ensuring relevance and SEO-friendliness.

### 6. **Fraud Detection and Prevention**
   - **Opportunity**: Utilize AI agents to analyze transaction patterns and identify anomalies indicating fraudulent activity.
   - **Benefit**: Enhances security for eCommerce platforms, protecting against losses and maintaining customer trust.

### 7. **User Onboarding and Tutorial Automation**
   - **Opportunity**: Create intelligent onboarding processes powered by AI that guide users through systems and features.
   - **Benefit**: Reduces customer drop-off rates during onboarding, ensuring users are effectively engaged from the start.

### 8. **Payment Processing Optimization**
   - **Opportunity**: Automate payment validation and processing by integrating AI with payment gateways.
   - **Benefit**: Streamlines financial transactions, reduces processing errors, and improves cash flow management.

### 9. **Predictive Analytics for Business Insights**
   - **Opportunity**: Utilize AI agents to perform advanced analytics on user data for sales forecasts and market insights.
   - **Benefit**: Supports strategic planning with data-driven insights, enhancing the ability to make informed business decisions.

### 10. **AI-Driven A/B Testing**
   - **Opportunity**: Enable AI agents to conduct A/B testing on marketing campaigns and website designs to identify the most effective strategies.
   - **Benefit**: Ensures that businesses are using the most effective tactics for user engagement and conversion rates.

### 11. **Seamless Integration with Cloud Services**
   - **Opportunity**: Deploy AI agents that facilitate the integration of various cloud services, optimizing workflows and data sharing.
   - **Benefit**: Enhances productivity by ensuring users can easily interact with multiple systems without manual interventions.

### 12. **Compliance and Risk Management**
   - **Opportunity**: Leverage AI for monitoring and ensuring compliance with regulations (e.g., GDPR, PCI DSS) in cloud and eCommerce.
   - **Benefit**: Reduces legal risks and potential fines while fostering a culture of compliance within the organization.

### Conclusion:
AI agents powered by LLMs present numerous opportunities in cloud computing and eCommerce by automating workflows, enhancing efficiency, and reducing manual labor. Businesses that effectively integrate these AI

capabilities can expect to achieve significant competitive advantages, driving innovation and improving customer satisfaction in a rapidly evolving digital landscape.

In [ ]:
aakash_prompt = """Businesses are embracing AI agents to transform workflows, enhance efficiency, and reduce manual effort. These agents leverage LLMs for dynamic decision-making, workflow automation, and seamless collaboration with human teams and existing systems. Examine the role of AI agents in automating diverse business processes for financial services. List and summarize the business opportunities in financial services that can benefit from LLM powered AI Agents."""

In [ ]:
aakash_response = model.invoke([HumanMessage(aakash_prompt)])
print(aakash_response.content)

AI agents powered by large language models (LLMs) are increasingly being adopted in the financial services sector to streamline workflows, improve decision-making, and enhance collaboration across various functions. Below are some key roles and opportunities for LLM-powered AI agents in automating diverse business processes in financial services:

### Roles of AI Agents in Financial Services
1. **Dynamic Decision-Making**: AI agents can analyze vast amounts of data and provide real-time insights, enabling financial institutions to make informed decisions quickly.

2. **Workflow Automation**: Routine tasks such as data entry, report generation, and transaction processing can be automated, which saves time and reduces errors.

3. **Customer Service and Engagement**: AI agents can interact with customers through chatbots or virtual assistants, providing support, answering inquiries, and personalizing services based on user interactions.

4. **Compliance and Risk Management**: AI can help monitor transactions and assess risks by identifying unusual patterns or behaviors and ensuring compliance with regulatory standards.

5. **Fraud Detection and Prevention**: AI agents are capable of identifying anomalies in financial transactions and flagging potential fraudulent activities for further investigation.

6. **Personalized Financial Advice**: Through analyzing customer data, AI can provide tailored financial recommendations, investment strategies, and optimization of financial portfolios.

### Business Opportunities in Financial Services Utilizing LLM-Powered AI Agents

1. **Enhanced Customer Support**:
   - **Description**: Implement chatbots and virtual assistants to handle customer inquiries and support.
   - **Opportunity**: 24/7 availability and rapid response times can significantly enhance customer satisfaction and reduce operational costs.

2. **Automated Financial Reporting**:
   - **Description**: Automate the generation of financial statements and reports based on real-time data.
   - **Opportunity**: Reduces manual effort, increases accuracy, and accelerates decision-making processes.

3. **Risk Assessment and Management**:
   - **Description**: Use AI to automate credit assessments and risk modeling.
   - **Opportunity**: More accurate risk profiling can lead to better lending decisions and minimized defaults.

4. **Fraud Detection Solutions**:
   - **Description**: Deploy AI to continuously monitor transactions for signs of fraudulent activity.
   - **Opportunity**: Early detection reduces potential losses and enhances security for both consumers and institutions.

5. **Regulatory Compliance Automation**:
   - **Description**: Automate compliance checks and reporting to regulatory bodies.
   - **Opportunity**: Streamlining compliance processes can reduce the cost of non-compliance and ensure adherence to evolving regulations.

6. **Personalized Banking Services**:
   - **Description**: Provide customized financial products and services based on customer behavior and preferences.
   - **Opportunity**: Increases customer retention and loyalty through tailored offerings, improving overall customer lifetime value.

7. **Market Analysis and Research**:
   - **Description**: Automate data collection and analysis for market trends and competitor activities.
   - **Opportunity**: Provides financial institutions with actionable insights to inform strategic decisions and stay competitive.

8. **Investment Portfolio Management**:
   - **Description**: Use AI-driven analysis to optimize investment portfolios based on market conditions and risk tolerance.
   - **Opportunity**: Enhanced decision-making capabilities can lead to better investment outcomes and client satisfaction.

9. **Loan Processing Automation**:
   - **Description**: Automate the loan application review process, including document validation and risk assessment.
   - **Opportunity**: Faster loan approvals can attract more customers while reducing operational burden.

10. **Tax Compliance and Filing**:
    - **Description**: Use AI to automate tax calculations, compliance checks, and document preparation.
    - **Opportunity**: Reduces the complexity and risk associated with tax filing for both individuals and organizations.

### Conclusion
By leveraging LLM-powered AI agents, financial services can unlock significant benefits across various operational areas, from enhancing customer experiences to improving efficiency and compliance. By focusing on these opportunities, financial institutions can not only streamline their processes but also position themselves for sustained growth and innovation in a rapidly evolving industry.